# Chicago Public Schools | SQL & Python Data Analysis

## Project overview
This portfolio case study analyzes a prepared subset of **Chicago Public Schools 2011–2012 Progress Report Cards**, alongside Chicago community census indicators. Using Python, Pandas, SQLite, and SQL, it examines variation in school attendance, safety scores, enrollment, and community hardship.

### Analytical objectives
1. Explore the structure and coverage of the schools dataset.
2. Identify schools with particularly high or low recorded safety and attendance scores.
3. Compare aggregated enrollment across community areas.
4. Connect school records with census hardship indicators using SQL.

**Tools:** Python, Pandas, SQLite, Jupyter Notebook, SQL magic.

**Data provenance:** The underlying public data come from the City of Chicago. The prepared CSV subsets and original exercises were supplied through IBM Skills Network educational materials. This portfolio adaptation retains the completed SQL queries and saved outputs; it is not presented as an independently commissioned research study.

**Public dataset:** [Chicago Public Schools Progress Report Cards (2011–2012)](https://data.cityofchicago.org/Education/Chicago-Public-Schools-Progress-Report-Cards-2011-/9xs2-f89t)


## 1. Environment and data preparation
Connect to SQLite, load SQL magic, and import the prepared public schools data.

In [7]:
import csv, sqlite3

con = sqlite3.connect("RealWorldData.db")
cur = con.cursor()

In [ ]:
import prettytable
prettytable.DEFAULT = "DEFAULT"

In [ ]:
%load_ext sql

In [10]:
%sql sqlite:///RealWorldData.db

In [11]:

import pandas

df = pandas.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoPublicSchools.csv")
df.to_sql("CHICAGO_PUBLIC_SCHOOLS_DATA", con, if_exists='replace', index=False, method="multi")

566

## 2. Explore the database schema

In [12]:
# type in your query to retrieve list of all tables in the database
%sql SELECT name FROM sqlite_master WHERE type='table'

 * sqlite:///RealWorldData.db
Done.


name
CHICAGO_PUBLIC_SCHOOLS_DATA


In [13]:
# type in your query to retrieve the number of columns in the CHICAGO_PUBLIC_SCHOOLS_DATA table
%sql SELECT count(name) FROM PRAGMA_TABLE_INFO('CHICAGO_PUBLIC_SCHOOLS_DATA');

 * sqlite:///RealWorldData.db
Done.


count(name)
78


In [14]:
# type in your query to retrieve all column names in the CHICAGO_PUBLIC_SCHOOLS_DATA table along with their datatypes and length
%sql SELECT name,type,length(type) FROM PRAGMA_TABLE_INFO('CHICAGO_PUBLIC_SCHOOLS_DATA');

 * sqlite:///RealWorldData.db
Done.


name,type,length(type)
School_ID,INTEGER,7
NAME_OF_SCHOOL,TEXT,4
"Elementary, Middle, or High School",TEXT,4
Street_Address,TEXT,4
City,TEXT,4
State,TEXT,4
ZIP_Code,INTEGER,7
Phone_Number,TEXT,4
Link,TEXT,4
Network_Manager,TEXT,4


## 3. School performance analysis
The following SQL queries investigate school types, safety scores, attendance, and college enrollment.

## Problems

### Problem 1

##### How many Elementary Schools are in the dataset?


In [15]:
%%sql SELECT COUNT(*)
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
WHERE "Elementary, Middle, or High School" = 'ES';

 * sqlite:///RealWorldData.db
Done.


COUNT(*)
462


### Problem 2

##### What is the highest Safety Score?


In [20]:
%%sql
SELECT MAX(Safety_Score)
FROM CHICAGO_PUBLIC_SCHOOLS_DATA;


 * sqlite:///RealWorldData.db
Done.


MAX(Safety_Score)
99.0


### Problem 3

##### Which schools have highest Safety Score?


In [22]:

%%sql
SELECT NAME_OF_SCHOOL, Safety_Score
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
WHERE Safety_Score = (
    SELECT MAX(Safety_Score)
    FROM CHICAGO_PUBLIC_SCHOOLS_DATA
);

 * sqlite:///RealWorldData.db
Done.


NAME_OF_SCHOOL,SAFETY_SCORE
Abraham Lincoln Elementary School,99.0
Alexander Graham Bell Elementary School,99.0
Annie Keller Elementary Gifted Magnet School,99.0
Augustus H Burley Elementary School,99.0
Edgar Allan Poe Elementary Classical School,99.0
Edgebrook Elementary School,99.0
Ellen Mitchell Elementary School,99.0
James E McDade Elementary Classical School,99.0
James G Blaine Elementary School,99.0
LaSalle Elementary Language Academy,99.0


### Problem 4

##### What are the top 10 schools with the highest "Average Student Attendance"?


In [23]:
%%sql
SELECT NAME_OF_SCHOOL, "Average_Student_Attendance"
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
ORDER BY "Average_Student_Attendance" DESC
LIMIT 10;

 * sqlite:///RealWorldData.db
Done.


NAME_OF_SCHOOL,AVERAGE_STUDENT_ATTENDANCE
John Charles Haines Elementary School,98.40%
James Ward Elementary School,97.80%
Edgar Allan Poe Elementary Classical School,97.60%
Orozco Fine Arts & Sciences Elementary School,97.60%
Rachel Carson Elementary School,97.60%
Annie Keller Elementary Gifted Magnet School,97.50%
Andrew Jackson Elementary Language Academy,97.40%
Lenart Elementary Regional Gifted Center,97.40%
Disney II Magnet School,97.30%
John H Vanderpoel Elementary Magnet School,97.20%


### Problem 5

##### Retrieve the list of 5 Schools with the lowest Average Student Attendance sorted in ascending order based on attendance


In [26]:
%sql SELECT Name_of_School, Average_Student_Attendance  \
     from CHICAGO_PUBLIC_SCHOOLS_DATA \
     order by Average_Student_Attendance \
     LIMIT 5;

 * sqlite:///RealWorldData.db
Done.


NAME_OF_SCHOOL,AVERAGE_STUDENT_ATTENDANCE
Velma F Thomas Early Childhood Center,None
Richard T Crane Technical Preparatory High School,57.90%
Barbara Vick Early Childhood & Family Center,60.90%
Dyett High School,62.50%
Wendell Phillips Academy High School,63.00%


### Problem 6

##### Now remove the '%' sign from the above result set for Average Student Attendance column


In [25]:
%%sql
SELECT
    NAME_OF_SCHOOL,
    REPLACE(Average_Student_Attendance, '%', '') AS Attendance
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
ORDER BY CAST(
    REPLACE(Average_Student_Attendance, '%', '')
    AS REAL
) ASC
LIMIT 5;

 * sqlite:///RealWorldData.db
Done.


NAME_OF_SCHOOL,Attendance
Velma F Thomas Early Childhood Center,None
Richard T Crane Technical Preparatory High School,57.90
Barbara Vick Early Childhood & Family Center,60.90
Dyett High School,62.50
Wendell Phillips Academy High School,63.00


### Problem 7

##### Which Schools have Average Student Attendance lower than 70%?


In [29]:
%sql SELECT Name_of_School, Average_Student_Attendance  \
     from CHICAGO_PUBLIC_SCHOOLS_DATA \
     where CAST ( REPLACE(Average_Student_Attendance, '%', '') AS DOUBLE ) < 70 \
     order by Average_Student_Attendance;

 * sqlite:///RealWorldData.db
Done.


NAME_OF_SCHOOL,AVERAGE_STUDENT_ATTENDANCE
Richard T Crane Technical Preparatory High School,57.90%
Barbara Vick Early Childhood & Family Center,60.90%
Dyett High School,62.50%
Wendell Phillips Academy High School,63.00%
Orr Academy High School,66.30%
Manley Career Academy High School,66.80%
Chicago Vocational Career Academy High School,68.80%
Roberto Clemente Community Academy High School,69.60%


### Problem 8

##### Get the total College Enrollment for each Community Area


In [30]:
%%sql
SELECT COMMUNITY_AREA_NAME,
       SUM(College_Enrollment) AS Total_Enrollment
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
GROUP BY COMMUNITY_AREA_NAME;

 * sqlite:///RealWorldData.db
Done.


COMMUNITY_AREA_NAME,Total_Enrollment
ALBANY PARK,6864
ARCHER HEIGHTS,4823
ARMOUR SQUARE,1458
ASHBURN,6483
AUBURN GRESHAM,4175
AUSTIN,10933
AVALON PARK,1522
AVONDALE,3640
BELMONT CRAGIN,14386
BEVERLY,1636


### Problem 9

##### Get the 5 Community Areas with the least total College Enrollment  sorted in ascending order


In [32]:
%%sql
SELECT COMMUNITY_AREA_NAME,
       SUM(College_Enrollment) AS Total_Enrollment
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
GROUP BY COMMUNITY_AREA_NAME
ORDER BY Total_Enrollment ASC
LIMIT 5;

 * sqlite:///RealWorldData.db
Done.


COMMUNITY_AREA_NAME,Total_Enrollment
OAKLAND,140
FULLER PARK,531
BURNSIDE,549
OHARE,786
LOOP,871


### Problem 10

##### List 5 schools with lowest safety score.


In [34]:
%%sql
SELECT NAME_OF_SCHOOL, Safety_Score
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
WHERE Safety_Score != 'None'
ORDER BY Safety_Score ASC
LIMIT 5;

 * sqlite:///RealWorldData.db
Done.


NAME_OF_SCHOOL,SAFETY_SCORE
Edmond Burke Elementary School,1.0
Luke O'Toole Elementary School,5.0
George W Tilton Elementary School,6.0
Foster Park Elementary School,11.0
Emil G Hirsch Metropolitan High School,13.0


## 4. Connecting school performance with census hardship
Load the census subset to compare school enrollment with community hardship indicators.

In [35]:
df = pandas.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCensusData.csv")
df.to_sql("CENSUS_DATA", con, if_exists='replace', index=False,method="multi")

78

### Problem 11

##### Get the hardship index for the community area of the school which has College Enrollment of 4368


In [38]:
%%sql
SELECT CD.hardship_index
FROM CENSUS_DATA CD, CHICAGO_PUBLIC_SCHOOLS_DATA CPS
WHERE CD.community_area_number = CPS.community_area_number
AND CPS.college_enrollment = 4368;

 * sqlite:///RealWorldData.db
Done.


HARDSHIP_INDEX
6.0


### Problem 12

##### Get the hardship index for the community area which has the highest value for College Enrollment


In [39]:
%%sql
SELECT community_area_number,
       community_area_name,
       hardship_index
FROM CENSUS_DATA
WHERE community_area_number IN (
    SELECT community_area_number
    FROM CHICAGO_PUBLIC_SCHOOLS_DATA
    ORDER BY college_enrollment DESC
    LIMIT 1
);

 * sqlite:///RealWorldData.db
Done.


COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,HARDSHIP_INDEX
5.0,North Center,6.0


## 5. Key Findings

The findings below are based on the **saved SQL outputs** in this notebook and refer to the historical dataset, not current school conditions.

- **School coverage:** The dataset contains 566 school records, including 462 elementary schools.
- **Safety scores:** The highest recorded safety score was 99, achieved by multiple schools. The five lowest non-null scores ranged from 1 to 13.
- **Student attendance:** John Charles Haines Elementary School had the highest attendance among the top ten displayed schools at 98.40%. Eight schools recorded attendance below 70%.
- **Community enrollment:** Oakland had the lowest summed `College_Enrollment` value (140), followed by Fuller Park (531) and Burnside (549).
- **Community hardship:** North Center had a hardship index of 6 and was associated with the school identified as having the highest `College_Enrollment` value.

## 6. Recommendations

- **Attendance monitoring:** Investigate the factors associated with attendance below 70% at the eight identified schools.
- **Safety assessment:** Examine schools with low safety scores to understand potential areas requiring attention.
- **Enrollment comparison:** Consider the number of schools and other relevant denominators before comparing total enrollment across communities.
- **Data quality:** Address missing attendance values and convert percentage strings into numeric values for reliable analysis.

## 7. Conclusion and Key Takeaways

This project used SQL and Python to explore historical Chicago Public Schools data and associated community-level information. The analysis showed variation in school attendance, safety scores, and recorded enrollment. In particular, eight schools had attendance below 70%, the highest recorded safety score was 99, and community-level enrollment totals differed across the areas examined. These results help identify where more detailed investigation could be useful, but do not establish why the differences occurred.

### Key Takeaways

- **SQL for analysis:** Filtering, sorting, aggregation, subqueries, and joins made it possible to answer specific questions about schools and their surrounding communities.
- **Data quality matters:** Missing attendance values and percentages stored as text must be handled carefully to avoid misleading rankings or calculations.
- **Context matters:** Community enrollment totals should not be compared without considering the number of schools and other relevant denominators. Likewise, community hardship indicators do not, by themselves, explain individual school outcomes.
- **Actionable next steps:** The attendance and safety results provide a starting point for further investigation, rather than evidence that a particular intervention will be effective.

Overall, the project demonstrates how structured SQL queries can turn school records into interpretable descriptive findings while emphasizing the importance of accurate reporting, appropriate comparisons, and responsible recommendations.
